## Import packages

In [ ]:
import os, sys
import pandas as pd
from concurrent.futures import ProcessPoolExecutor, as_completed
import numpy as np
from scipy.sparse import coo_matrix
import cooler
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pylab as plt
import matplotlib as mpl
mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Arial'
mpl.rcParams['figure.dpi'] = 80
mpl.rcParams['savefig.dpi']=300

## Parameters

In [ ]:
workdir = "/home/x-wding2/project_wubin/BG/HIC/BICAN_pseudobulk"
figdir=None
mode='raw'
res=100000
chrom_size_path = "/home/x-wding2/Ref/hg38/hg38_ucsc.nochrM.sizes"
metadata_path = "/home/x-wding2/Projects/BG/clustering/100kb/cell_metadata_with_annotation.tsv"
query_str=None
adata_prefix="~/Projects/BG/adata/BG"
atac_adata_path=None
atac_ann_path=None
group_col='Subclass'
parent_col="Class"
sort_cols=None
stratify_figdir=None
stratify_col = None # Subclass
palette_path = "/home/x-wding2/Projects/BG/metadata/HMBA_color_palette.xlsx"
cpu = 16
obs_dim='cell'
var_dim="100kb"
mc_types = ["CGN", "CHN"]
kernel_name = "m3c"
excluded_groups=['Low quality cells','doublet','Unk']

query_str = "Study=='BICAN'"
# group_col = "Region"
# parent_col = 'Class'
# stratify_col = "Subclass"

In [ ]:
# Parameters
workdir = "/home/x-wding2/Projects/BG/HIC/BG_Class"
metadata_path = "/home/x-wding2/Projects/BG/clustering/100kb/annotations_without_Glut.tsv"
adata_prefix = "/home/x-wding2/Projects/BG/adata/BG"
atac_adata_path = "/home/x-wding2/Projects/BG/atac/HMBA.atac-5kb.h5ad"
atac_ann_path = "/home/x-wding2/Projects/BG/atac/obs.tsv"
pseudobulk_mcds_path = "/home/x-wding2/Projects/BG/pseudobulk/Class/Class.mcds"
pseudobulk_atac_adata_path = "/home/x-wding2/Projects/BG/atac/Pseudobulk.Class.5kb.h5ad"
chrom_size_path = "/home/x-wding2/Ref/hg38/hg38_ucsc.nochrM.sizes"
palette_path = "/home/x-wding2/Projects/BG/metadata/BG_color_palette.xlsx"
cg_count_path = "/home/x-wding2/Ref/hg38/annotations/CG-count-100k.tsv"
gene_annotation = "/home/x-wding2/Ref/hg38/annotations/gene_2000_2000.bed.gz"
marker_path = "/home/x-wding2/Projects/BG/metadata/BG_markers.xlsx"
marker_sheet = "BG"
mode = "raw"
prefix = "Class"
group_col = "Class"
parent_col = "Neighborhood"
sort_cols = ["Neighborhood"]
stratify_col = None
loop_prefix = None
query_str = None
excluded_groups = ["Low quality cells", "doublet", "Unk", "Glut"]
figdir = None
cpu = 8
indir = None
kernel_name = "m3c"
genome = "hg38"
obs_dim = "cell"
var_dim = "100kb"
var_dim5kb = "5kb"
extract_group = None
query_group = None
mc_types = ["CGN", "CHN"]
normalize_per_cell = True
split_order = ["Nonneuron", "Subpallium GABA", "Glut Sero Dopa", "Subpallium GABA-Glut"]
diff_comp_dir = None
dchic_env = "dchic"
param_yaml = "/home/x-wding2/Projects/Github/pym3c/pym3c/notebooks/templates/BG_Subclass_hic.yaml"
outdir = "/home/x-wding2/Projects/BG/HIC/BG_Class/compartment"
outname = "2.Class.compartment_comethylation.ipynb"


In [ ]:
if not workdir is None:
    workdir=os.path.expanduser(workdir)
if figdir is None:
    assert not workdir is None
    figdir=os.path.join(workdir,"compartment",f"{mode}_{group_col}")
else:
    figdir=os.path.expanduser(figdir)
if not os.path.exists(figdir):
    os.makedirs(figdir, exist_ok=True)
chrom_size_path=os.path.expanduser(chrom_size_path)
metadata_path=os.path.expanduser(metadata_path)
chrom_size_path=os.path.expanduser(chrom_size_path)
adata_prefix=os.path.expanduser(adata_prefix)
print(figdir,adata_prefix,group_col)

In [ ]:
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
chrom_sizes = chrom_sizes[chrom_sizes.index!='chrY']
df_chr=chrom_sizes.to_frame()
df_chr['N']=df_chr.index.to_series().apply(lambda x:int(x.lstrip('chr')) if x.lstrip('chr') not in ['X','Y'] else 30) #sort by chrom int (1,2,3..)
df_chr.sort_values('N',inplace=True)
chrom_sizes=chrom_sizes.loc[df_chr.index.tolist()]
if metadata_path.endswith('.csv') or metadata_path.endswith('.csv.gz'):
    metadata = pd.read_csv(metadata_path, index_col=0)
else:
    metadata = pd.read_csv(metadata_path, index_col=0,sep='\t')
metadata=metadata.loc[~ (metadata[group_col].isin(excluded_groups))]
metadata=metadata.loc[metadata[group_col].apply(lambda x:not str(x).endswith('-Unk'))]
if not stratify_col is None:
    metadata=metadata.loc[~ (metadata[stratify_col].isin(excluded_groups))]
if not parent_col is None:
    metadata=metadata.loc[~ (metadata[parent_col].isin(excluded_groups))]
if sort_cols is None:
    sort_cols=parent_col
if not query_str is None:
    print(query_str)
    metadata=metadata.query(query_str)
print(metadata.shape)
metadata.head()

In [ ]:
chrom_sizes

In [ ]:
# read transformed X_pca for each cell type generated by 1.call_compartment.ipynb
comp = pd.read_csv(os.path.join(figdir,"compartment_score_pcs.tsv.gz"),sep='\t',index_col=0)
print(comp.shape)
comp.head()

## Define group_order and colors

In [ ]:
# define group_order (such as cell types) and stratify2group (such as Subclass to Subclass+Region)
if not sort_cols is None:
    metadata.sort_values(sort_cols,inplace=True)
else:
    metadata.sort_values(group_col,inplace=True)

group_order=comp.columns.tolist()
if not stratify_col is None:
    if stratify_figdir is None:
        metadata['GroupID']=metadata[stratify_col].map(str)+'+'+metadata[group_col].map(str) #apply(lambda x:x.replace(' ','_')).
    else:
        metadata['GroupID']=metadata[group_col]
    stratify2group=metadata.loc[:,[stratify_col,'GroupID']].drop_duplicates().groupby(stratify_col)['GroupID'].apply(lambda x:x.unique().tolist()).to_dict()
    # group2stratify=metadata.loc[:,[stratify_col,'GroupID']].drop_duplicates().set_index('GroupID')[stratify_col].to_dict()
    if not parent_col is None:
        group2parent=metadata.loc[:,[parent_col,'GroupID']].drop_duplicates().set_index('GroupID')[parent_col].to_dict()
    else:
        group2parent={} # group is either cell types (such ODC) or GroupID (such as ODC+Pu), parent is Class (such as Glut, Nonneuron)
else:
    stratify2group={}
    # group2stratify={}
    group2parent=metadata.loc[:,[parent_col,group_col]].drop_duplicates().set_index(group_col)[parent_col].to_dict()
group2parent['merged']='merged'
print(group_order)
print(stratify2group)
# print(group2stratify)
print(group2parent)

In [ ]:
# read color palette
D=pd.read_excel(os.path.expanduser(palette_path),
                sheet_name=None, index_col=0)
group_colors=D[group_col].Hex.to_dict()
if not parent_col is None:
    parent_colors=D[parent_col].Hex.to_dict()
    parent_colors['merged']='black'
else:
    parent_colors=None
if not stratify_col is None:
    D1=D[stratify_col].Hex.to_dict()
    stratify_colors={k: D1[k] for k in D1 if k in stratify2group}
else:
    stratify_colors={}
print(group_colors)
print(parent_colors)
print(stratify_colors)

## Calculate 100kb mC correlation

In [ ]:
bins_df = cooler.binnify(pd.Series(chrom_sizes), res)
print(bins_df.shape)
bins_df.head()

In [ ]:
def get_chrom_offsets(bins_df):
    chrom_offset = {chrom: bins_df[bins_df['chrom'] == chrom].index[0]
                    for chrom in bins_df['chrom'].cat.categories}
    return chrom_offset # dict of bin ID for each chrom bins

def chrom_iterator(adata,chrom_sizes,offset):
    for i,chrom in enumerate(chrom_sizes.keys()):
        n_bins = int(chrom_sizes[chrom] // res) + 1 # 1-based
        selb = (adata.var['chrom']==chrom).values
        # calculate pearson correlation for 100-kb bins.
        matrix = np.corrcoef(adata.X[:, selb].T) #correlation for rows, Each row of `x` represents a variable, and each column a single observation of all those variables
        matrix[np.isnan(matrix)] = 0 # fillna with 0 (correlation)
        matrix = coo_matrix(np.triu(matrix)) #convert to sparse matrix

        pixel_df = pd.DataFrame({
            'bin1_id': matrix.row,
            'bin2_id': matrix.col,
            'count': matrix.data
        })
        pixel_df.iloc[:, :2] += offset[chrom] #convert into the ID of bins.
        yield pixel_df

In [ ]:
from pym3c.clustering import normalize_mc_by_cell
from pym3c.adata import merge_adata_regions
import anndata
def calculate_100kb_mc_correlation(chrom_sizes,res,use_cells,adata_prefix,mc_type,cool_uri):
    adata=anndata.read_h5ad(f"{adata_prefix}.100kb-{mc_type}.h5ad",backed='r')
    bins_df = cooler.binnify(pd.Series(chrom_sizes), res)
    offset = get_chrom_offsets(bins_df) #the ID of bins, for example, the first ID of 100kb bin for chr2 is 1955, 0-based
    adata1=normalize_mc_by_cell(adata[use_cells,:].to_memory(), normalize_per_cell=True,clip_norm_value=None,verbose=0)
    adata.file.close()
    adata1.var['chrom']=adata1.var_names.to_series().apply(lambda x:x.split('_')[0])
    chrom_iter = chrom_iterator(adata=adata1,chrom_sizes=chrom_sizes,offset=offset)
    cooler.create_cooler(cool_uri=cool_uri,bins=bins_df,pixels=chrom_iter,
                         ordered=True,dtypes={'count': np.float32})
    return cool_uri

def calculate_100kb_atac_correlation(chrom_sizes,res,use_cells,atac_adata_path,cool_uri):
    raw_adata=anndata.read_h5ad(os.path.expanduser(atac_adata_path),backed='r')
    adata1=raw_adata[use_cells,:].to_memory()
    # adata=adata.raw.to_adata()
    raw_adata.file.close()
    data=merge_adata_regions(adata1,bin_size=5000,use_raw=False,
    		res=res,filter_chroms=True,exclude_chroms=['chrY','chrM'])
    adata1 = anndata.AnnData(X=data)
    bins_df = cooler.binnify(pd.Series(chrom_sizes), res)
    offset = get_chrom_offsets(bins_df) #the ID of bins, for example, the first ID of 100kb bin for chr2 is 1955, 0-based
    adata1.var['chrom']=adata1.var_names.to_series().apply(lambda x:x.split('_')[0])
    chrom_iter = chrom_iterator(adata=adata1,chrom_sizes=chrom_sizes,offset=offset)
    # print(adata1.shape)
    cooler.create_cooler(cool_uri=cool_uri,bins=bins_df,pixels=chrom_iter,
                         ordered=True,dtypes={'count': np.float32})
    return cool_uri

In [ ]:
if stratify_col is None or stratify_figdir is not None:
    group_name=group_col
else:
    group_name="GroupID"
downsample=1500
print(group_name)

with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    for mc_type in mc_types:
        mc_corr_outdir=os.path.join(workdir,"compartment",f"{group_col}.{mc_type}.{var_dim}.correlations")
        if not os.path.exists(mc_corr_outdir):
            os.makedirs(mc_corr_outdir,exist_ok=True)
        adata=anndata.read_h5ad(f"{adata_prefix}.100kb-{mc_type}.h5ad",backed='r')
        common_cells=list(set(adata.obs_names) & set(metadata.index.tolist()))
        adata.file.close()
        for ct in metadata[group_name].unique().tolist()+['merged']:
            ct_name=ct.replace(' ','_')
            cool_uri=os.path.join(mc_corr_outdir,f"{ct_name}.{mc_type}.{var_dim}.corr.cool")
            if os.path.exists(cool_uri):
                continue
            metadata1=metadata.loc[common_cells] #query_str has been applied to metadata, such as Study=='HBA'
            if ct!='merged':
                metadata1=metadata1.loc[metadata1[group_name]==ct]
                if metadata1.shape[0] > downsample:
                    use_cells=metadata1.sample(downsample).index.tolist()
                else:
                    use_cells=metadata1.index.tolist()
            else:
                if metadata1.shape[0] > downsample:
                    use_cells=metadata1.sample(downsample).index.tolist()
                else:
                    use_cells=metadata1.index.tolist()
            # calculate_100kb_mc_correlation(chrom_sizes,res,use_cells,adata_prefix,mc_type,cool_uri)
            future = executor.submit(
                calculate_100kb_mc_correlation,
                chrom_sizes=chrom_sizes,res=res,use_cells=use_cells,
                adata_prefix=adata_prefix,mc_type=mc_type,cool_uri=cool_uri
            )
            futures[future] = f"{mc_type}-{ct}"
    # ATAC
    if not atac_adata_path is None:
        atac_corr_outdir=os.path.join(workdir,"compartment",f"{group_col}.ATAC.correlations")
        if not os.path.exists(atac_corr_outdir):
            os.makedirs(atac_corr_outdir,exist_ok=True)
        adata=anndata.read_h5ad(os.path.expanduser(atac_adata_path),backed='r')
        obs=adata.obs.copy()
        adata.file.close()
        ann=pd.read_csv(os.path.expanduser(atac_ann_path),sep='\t',index_col=0)
        common_cells=list(set(obs.index.tolist()) & set(ann.index.tolist()))
        ann=ann.loc[common_cells] #query_str has been applied to metadata, such as Study=='HBA'
        for ct in ann[group_name].unique().tolist()+['merged']:
            ct_name=ct.replace(' ','_')
            cool_uri=os.path.join(atac_corr_outdir,f"{ct_name}.ATAC.corr.cool")
            if os.path.exists(cool_uri):
                continue
            if ct!='merged':
                ann1=ann.loc[ann[group_name]==ct]
                if ann1.shape[0] > downsample:
                    use_cells=ann1.sample(downsample).index.tolist()
                else:
                    use_cells=ann1.index.tolist()
            else:
                if ann.shape[0] > downsample:
                    use_cells=ann.sample(downsample).index.tolist()
                else:
                    use_cells=ann.index.tolist()
            # print(ct,len(use_cells))
            # calculate_100kb_atac_correlation(chrom_sizes,res,use_cells,atac_adata_path,cool_uri)
            future = executor.submit(
                calculate_100kb_atac_correlation,
                chrom_sizes=chrom_sizes,res=res,use_cells=use_cells,
                atac_adata_path=atac_adata_path,cool_uri=cool_uri
            )
            futures[future] = f"ATAC-{ct}"
    for future in as_completed(futures):
        ct = futures[future]
        r = future.result()
        print(ct,end=',')

In [ ]:
def saddle(cool, ct_comp):
    sad = np.zeros((50, 50))
    count = np.zeros((50, 50))
    for chrom in chrom_sizes.index:
        C = cool.matrix(balance=False, sparse=True).fetch(chrom).toarray() #correlations
        C = C - np.diag(np.diag(C))
        pc = ct_comp.loc[(ct_comp.index.str.split('-').str[0]==chrom)]
        pc.index = [int(xx.split('-')[1]) for xx in pc.index]
        C = C[np.ix_(pc.index, pc.index)]
        labels, groups = pd.qcut(pc, 50, labels=False, retbins=True)
        sad += np.array([[C[np.ix_(labels==i, labels==j)].sum() for i in range(50)] for j in range(50)])
        count += np.array([[(labels==i).sum()*(labels==j).sum() for i in range(50)] for j in range(50)])
    return sad/count

def plot_saddle(cell_types,saddle_dict,stratify_col,
                parent_colors,group2parent,parent_col,group_colors,
                figdir,stratify=None,vmax=0.03,prefix="test"):
    ncols=len(cell_types)
    nrows=int(np.ceil(len(cell_types)/ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*3,nrows*2.4), gridspec_kw={'wspace':0.05,'hspace':0.0}, dpi=300)
    # fig.subplots_adjust(hspace=0.01, wspace=0.01)
    for i,ct in enumerate(cell_types): # if stratify_col is None, ct is cell type, otherwise, ct is GroupID (such as ODC+Pu)
        if isinstance(axes,np.ndarray):
            ax = axes.flatten()[i]
        else:
            ax=axes
        # vmax=max([abs(np.max(saddle_dict[ct])),abs(np.min(saddle_dict[ct]))])
        # plot = ax.imshow(np.log2(saddle_dict[ct]+0.001), cmap='coolwarm', vmin=vmin, vmax=vmax)
        plot = ax.imshow(saddle_dict[ct], cmap='coolwarm',vmin=-vmax,vmax=vmax)
        ax.set_xticks([])
        ax.set_yticks([])
        if stratify_col is None: # group is cell type and parent is Class
            color=parent_colors[group2parent[ct]] if not parent_col is None else 'black'
            title=ct
        else:
            g2=ct.split('+')[1] if stratify_figdir is None else ct
            # color=stratify_colors[group2stratify[ct]]
            color=group_colors[g2]
            title=g2
        ax.set_title(title, fontsize=10,color=color,fontweight='bold')
        cbar = plt.colorbar(plot, ax=ax, shrink=0.3, fraction=0.3,aspect=5)
        plot.set_clim(np.around([cbar.vmin, cbar.vmax], decimals=2))
        cbar.set_ticks([cbar.vmin,(cbar.vmin+cbar.vmax)/2, cbar.vmax])
        # cbar.set_label('mC corr')
    if isinstance(axes,np.ndarray):
        for ax in axes.flatten()[len(cell_types):]:
            ax.axis('off')
    # plt.tight_layout()
    if stratify is None:
        outname=os.path.join(figdir,f'{prefix}.corr.saddle.pdf')
    else:
        outname=os.path.join(figdir,f'{stratify}.{prefix}.corr.saddle.pdf')
    plt.savefig(outname,transparent=True,bbox_inches='tight')
    plt.show()

In [ ]:
for mc_type in mc_types:
    mc_corr_outdir=os.path.join(workdir,"compartment",f"{group_col}.{mc_type}.{var_dim}.correlations")
    if not os.path.exists(os.path.join(figdir,f'{mc_type}.{var_dim}.corr.saddle.npz')):
        saddle_dict={}
        with ProcessPoolExecutor(cpu) as executor:
            futures = {}
            for t,ct in enumerate(group_order):
                ct_name=ct.replace(' ','_')
                cool_uri=os.path.join(mc_corr_outdir,f"{ct_name}.{mc_type}.{var_dim}.corr.cool")
                cool = cooler.Cooler(cool_uri)
                future = executor.submit(
                    saddle,
                    cool=cool,
                    ct_comp=comp[ct],
                )
                futures[future] = ct

            for future in as_completed(futures):
                ct = futures[future]
                sad = future.result()
                saddle_dict[ct]=sad
                print(ct,end=',')
        np.savez_compressed(os.path.join(figdir,f'{mc_type}.{var_dim}.corr.saddle.npz'), **saddle_dict)
    else:
        saddle_dict=np.load(os.path.join(figdir,f'{mc_type}.{var_dim}.corr.saddle.npz'), allow_pickle=True)

    vmax=0.02 if mc_type=='CGN' else 0.03
    prefix=f"{mc_type}.{var_dim}"
    if stratify_col is None:
        plot_saddle(group_order,saddle_dict,stratify_col,
                parent_colors,group2parent,parent_col,group_colors,
                figdir,stratify=None,vmax=vmax,prefix=prefix)
    else:
        for stratify in stratify2group:
            cell_types=[ct for ct in stratify2group[stratify] if ct in saddle_dict]
            if len(cell_types) <= 1:
                continue
            plot_saddle(cell_types,saddle_dict,stratify_col,
                parent_colors,group2parent,parent_col,group_colors,
                figdir,stratify=stratify,vmax=vmax,prefix=prefix)

In [ ]:
# ATAC
if not atac_adata_path is None:
    atac_corr_outdir=os.path.join(workdir,"compartment",f"{group_col}.ATAC.correlations")
    if not os.path.exists(os.path.join(figdir,'ATAC.corr.saddle.npz')):
        saddle_dict={}
        with ProcessPoolExecutor(cpu) as executor:
            futures = {}
            for t,ct in enumerate(group_order):
                ct_name=ct.replace(' ','_')
                cool_uri=os.path.join(atac_corr_outdir,f"{ct_name}.ATAC.corr.cool")
                if not os.path.exists(cool_uri):
                    continue
                cool = cooler.Cooler(cool_uri)
                future = executor.submit(
                    saddle,
                    cool=cool,
                    ct_comp=comp[ct],
                )
                futures[future] = ct
    
            for future in as_completed(futures):
                ct = futures[future]
                sad = future.result()
                saddle_dict[ct]=sad
                print(ct,end=',')
        np.savez_compressed(os.path.join(figdir,'ATAC.corr.saddle.npz'), **saddle_dict)
    else:
        saddle_dict=np.load(os.path.join(figdir,'ATAC.corr.saddle.npz'), allow_pickle=True)
    
    vmax=0.4
    prefix=f"ATAC"
    if stratify_col is None:
        plot_saddle(group_order,saddle_dict,stratify_col,
                parent_colors,group2parent,parent_col,group_colors,
                figdir,stratify=None,vmax=vmax,prefix=prefix)
    else:
        for stratify in stratify2group:
            cell_types=[ct for ct in stratify2group[stratify] if ct in saddle_dict]
            if len(cell_types) <= 1:
                continue
            plot_saddle(cell_types,saddle_dict,stratify_col,
                parent_colors,group2parent,parent_col,group_colors,
                figdir,stratify=stratify,vmax=vmax,prefix=prefix)